## Ensemble Engineer


Se implementará técnicas de VotingClassfier (hard y soft voting) usando los modelos que mejor se han desempeñado en el baseline:

* Random Forest

* Decisition Tree

* XGBoost



In [3]:
import joblib
import pandas as pd

from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [4]:
# =========================================================
# Cargar datasets
# =========================================================

X_train = pd.read_csv("../data/processed/X_train_bal.csv")
y_train = pd.read_csv("../data/processed/y_train_bal.csv").squeeze()

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [5]:
# =========================================================
# Cargar modelos entrenados
# =========================================================

best_xgb = joblib.load("../models/baseline_xgb.pkl")
best_dt  = joblib.load("../models/baseline_dt.pkl")
best_rf  = joblib.load("../models/baseline_rf.pkl")

In [8]:
# =========================================================
# Configuración CV y métricas
# =========================================================

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [9]:
# =========================================================
# HARD VOTING
# =========================================================

hard_voting = VotingClassifier(
    estimators=[
        ('dt', best_dt),
        ('rf', best_rf),
        ('xgb', best_xgb)
    ],
    voting='hard'
)

# Cross-validation
hard_scores = cross_validate(
    hard_voting,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

print("=== HARD VOTING CV RESULTS ===")
for metric in scoring.keys():
    print(
        f"{metric}: "
        f"{hard_scores[f'test_{metric}'].mean():.4f}"
    )

# Entrenamiento final
hard_voting.fit(X_train, y_train)

# Predicción
y_pred_hard = hard_voting.predict(X_test)

print("\n=== HARD VOTING TEST REPORT ===")
print(classification_report(y_test, y_pred_hard))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_hard))

=== HARD VOTING CV RESULTS ===
accuracy: 0.8819
precision: 0.8929
recall: 0.8679
f1: 0.8802

=== HARD VOTING TEST REPORT ===
              precision    recall  f1-score   support

           0       0.89      0.91      0.90     15002
           1       0.83      0.81      0.82      8839

    accuracy                           0.87     23841
   macro avg       0.86      0.86      0.86     23841
weighted avg       0.87      0.87      0.87     23841


Confusion Matrix:
[[13587  1415]
 [ 1722  7117]]


In [10]:
# =========================================================
# SOFT VOTING
# =========================================================

soft_voting = VotingClassifier(
    estimators=[
        ('dt', best_dt),
        ('rf', best_rf),
        ('xgb', best_xgb)
    ],
    voting='soft',

    # Más peso a RF y XGB
    weights=[1, 3, 3]
)

# Cross-validation
soft_scores = cross_validate(
    soft_voting,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

print("\n==============================")
print("SOFT VOTING - CV RESULTS")
print("==============================")

for metric in scoring.keys():
    print(
        f"{metric}: "
        f"{soft_scores[f'test_{metric}'].mean():.4f}"
    )

# Entrenamiento final
soft_voting.fit(X_train, y_train)

# Predicciones
y_pred_soft = soft_voting.predict(X_test)

# Probabilidades
y_prob_soft = soft_voting.predict_proba(X_test)[:, 1]

print("\n==============================")
print("SOFT VOTING - TEST REPORT")
print("==============================")

print(classification_report(y_test, y_pred_soft))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_soft))

# ROC-AUC
auc_soft = roc_auc_score(y_test, y_prob_soft)

print(f"\nSoft Voting ROC-AUC: {auc_soft:.4f}")


SOFT VOTING - CV RESULTS
accuracy: 0.8808
precision: 0.8913
recall: 0.8673
f1: 0.8791

SOFT VOTING - TEST REPORT
              precision    recall  f1-score   support

           0       0.89      0.90      0.90     15002
           1       0.83      0.81      0.82      8839

    accuracy                           0.87     23841
   macro avg       0.86      0.86      0.86     23841
weighted avg       0.87      0.87      0.87     23841


Confusion Matrix:
[[13570  1432]
 [ 1704  7135]]

Soft Voting ROC-AUC: 0.9344


In [ ]:
# =========================================================
# Guardar modelos ensemble
# =========================================================

joblib.dump(
    hard_voting,
    "../models/hard_voting.pkl"
)

joblib.dump(
    soft_voting,
    "../models/soft_voting.pkl"
)

print("\nVoting classifiers guardados correctamente.")

## Construcción de un Stacking Classifier

Para la construcción del stacking classifier trabajaremos con los siguientes modelos:

* Random Forest

* Decisition Tree

* XGBoost

In [1]:
import pandas as pd
import joblib

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    roc_auc_score
)

In [2]:
# =========================================================
# LOAD DATA
# =========================================================

X_train = pd.read_csv("../data/processed/X_train_bal.csv")
y_train = pd.read_csv("../data/processed/y_train_bal.csv").squeeze()

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [3]:
# =========================================================
# LOAD BEST BASE MODELS
# =========================================================

best_xgb = joblib.load("../models/baseline_xgb.pkl")
best_dt  = joblib.load("../models/baseline_dt.pkl")
best_rf  = joblib.load("../models/baseline_rf.pkl")

In [4]:
# =========================================================
# META-LEARNER
# =========================================================

meta_learner = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [5]:

# =========================================================
# STACKING CLASSIFIER
# =========================================================

stack_model = StackingClassifier(
    estimators=[
        ("xgb", best_xgb),
        ("dt", best_dt),
        ("rf", best_rf)
    ],
    
    final_estimator=meta_learner,

    # Usa probabilidades como input del meta-modelo
    stack_method="predict_proba",

    # Cross-validation interna
    cv=5,

    # Usa predicciones originales + probabilidades
    passthrough=False,

    n_jobs=-1
)

In [6]:
# =========================================================
# TRAIN
# =========================================================

print("Training Stacking Classifier...")

stack_model.fit(X_train, y_train)

print("Training completed.")

Training Stacking Classifier...
Training completed.


In [7]:
# =========================================================
# PREDICTIONS
# =========================================================

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]

In [8]:
# =========================================================
# EVALUATION
# =========================================================

print("\n==============================")
print("STACKING CLASSIFIER RESULTS")
print("==============================\n")

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_prob):.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


STACKING CLASSIFIER RESULTS

Accuracy : 0.8695
F1 Score : 0.8216
ROC AUC  : 0.9363

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.90      0.90     15002
           1       0.83      0.81      0.82      8839

    accuracy                           0.87     23841
   macro avg       0.86      0.86      0.86     23841
weighted avg       0.87      0.87      0.87     23841


Confusion Matrix:

[[13563  1439]
 [ 1673  7166]]


In [ ]:
# =========================================================
# SAVE MODEL
# =========================================================

joblib.dump(
    stack_model,
    "../models/stacking_classifier.pkl"
)

print("\nStacking model saved successfully.")